# Track 2 · Stage 1+2 — Generate the synthetic code-mixed corpus

Replication of Biswas et al., Interspeech 2025 (Track 2).

This notebook does **all** data generation and pushes the result to the Hub:

1. **Stage 1** — few-shot prompt Llama-3.1-8B for Hindi-English bigrams, filter them,
   expand each into four sentences (~16k sentences).
2. **Stage 2** — synthesize them with Indic Parler-TTS (Rohit / Divya), 22h of audio.
3. Push `synth_t2` + `t1_ids.json` to `RohanRamesh/hi-en-synth-cs`.

### Why one notebook can host both models
`parler-tts` hard-pins `transformers==4.46.1`. vLLM will not tolerate that pin, but
`transformers` 4.46.1 *does* support Llama-3.1, so the LLM runs under bitsandbytes NF4
in the same environment. (vLLM is also out on principle: its AWQ kernels need compute
capability ≥ 8.0 and the T4 is sm75.)

### Before you run
* Accept the licence for `ai4bharat/indic-parler-tts` (gated) and `meta-llama/Llama-3.1-8B-Instruct`.
* Add your HF **write** token to Kaggle Secrets as `HF_TOKEN`.
* Turn notebook internet **on**, accelerator **GPU T4 x2**.
* `STAGE` below lets you resume: `text`, `audio`, or `both`.

Runtime ≈ 2h (LLM) + 4–6h (TTS). Everything is cached/sharded, so a 12h timeout
costs at most the in-flight shard.

## 0 · Install

**Cell order matters.** The `transformers` downgrade silently no-ops if anything has already imported `transformers`. Do not import it above this cell.

In [ ]:
!pip install -q "transformers==4.46.1" "datasets<4" bitsandbytes accelerate soxr
!pip install -q git+https://github.com/huggingface/parler-tts.git
!pip install -q git+https://github.com/BRUH-MAIN/codeswitching.git

import transformers
assert transformers.__version__ == "4.46.1", (
    f"transformers is {transformers.__version__}, expected 4.46.1 - "
    "something imported it before the downgrade took effect. Restart the kernel."
)
print("transformers", transformers.__version__)

In [ ]:
import os, gc, json, subprocess, sys
from pathlib import Path

# Keep 8.7 GB of model weights out of the 20 GB /kaggle/working budget.
os.environ["HF_HOME"] = "/kaggle/temp/hf"
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN)

STAGE      = "both"          # "text" | "audio" | "both"
REAL_REPO  = "RohanRamesh/mucs-he-cs"
SYNTH_REPO = "RohanRamesh/hi-en-synth-cs"
LLM        = "meta-llama/Llama-3.1-8B-Instruct"   # or unsloth/Meta-Llama-3.1-8B-Instruct (ungated)
NUM_SHARDS = 4

WORK = Path("/kaggle/working")
MAN  = WORK / "manifests"; MAN.mkdir(parents=True, exist_ok=True)
CACHE = WORK / "llm_cache"; CACHE.mkdir(parents=True, exist_ok=True)
AUDIO = WORK / "audio";    AUDIO.mkdir(parents=True, exist_ok=True)

def run(*args):
    print(">", " ".join(str(a) for a in args), flush=True)
    subprocess.run([sys.executable, "-m", *args], check=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Pull the in-domain transcripts (few-shot exemplars)

Track 2 never trains on real code-switched audio — we only need the *text* of the MUCS train split.

In [ ]:
from datasets import load_dataset
from csasr.manifest import write_jsonl

train_text = load_dataset(REAL_REPO, "train_text", split="train", token=HF_TOKEN)
write_jsonl(MAN / "mucs_train.jsonl", [dict(r) for r in train_text])
print(f"{len(train_text):,} in-domain sentences for few-shot prompting")
print(train_text[0]["text"])

## 2 · Stage 1a — generate bigrams

Paper: 44,657 raw → 5,932 unique (13.3%).

In [ ]:
if STAGE in ("text", "both"):
    run("csasr.llm.gen_bigrams",
        "--train-manifest", MAN / "mucs_train.jsonl",
        "--out", MAN / "bigrams_raw.jsonl",
        "--cache", CACHE / "bigrams.jsonl",
        "--model", LLM, "--n-calls", "4466", "--batch-size", "16")

## 3 · Stage 1b — filter

Deterministic script filter, then an LLM translation check with 3-sample self-consistency.
Paper: 5,932 unique → 5,477 valid (92.3%).

In [ ]:
if STAGE in ("text", "both"):
    run("csasr.llm.filter_bigrams",
        "--raw", MAN / "bigrams_raw.jsonl",
        "--out", MAN / "bigrams_valid.jsonl",
        "--cache", CACHE / "transcheck.jsonl",
        "--model", LLM, "--items-per-call", "20", "--n-samples", "3")

## 4 · Stage 2a — expand bigrams into sentences

Four per bigram (2 English-matrix, 2 Hindi-matrix). Paper: ~16,000 unique from a theoretical 21,908.

In [ ]:
if STAGE in ("text", "both"):
    run("csasr.llm.gen_sentences",
        "--bigrams", MAN / "bigrams_valid.jsonl",
        "--out", MAN / "sentences.jsonl",
        "--cache", CACHE / "sentences.jsonl",
        "--model", LLM, "--batch-size", "16")

### GATE 1 — yield ratios must track the paper

A large divergence in the 13.3% dedup rate means the prompt or temperature is off. Decide here, not later.

In [ ]:
from csasr.manifest import read_jsonl

raw   = list(read_jsonl(MAN / "bigrams_raw.jsonl"))
uniq  = {r["bigram"] for r in raw}
valid = list(read_jsonl(MAN / "bigrams_valid.jsonl"))
sents = list(read_jsonl(MAN / "sentences.jsonl"))

rows = [
    ("raw bigrams",    len(raw),   44_657, None),
    ("unique bigrams", len(uniq),   5_932, len(uniq) / max(len(raw), 1)),
    ("valid bigrams",  len(valid),  5_477, len(valid) / max(len(uniq), 1)),
    ("sentences",      len(sents), 16_000, None),
]
print(f"{'metric':<16}{'ours':>10}{'paper':>10}{'survival':>12}   paper survival")
for name, got, want, surv in rows:
    s = f"{surv:.1%}" if surv else "-"
    print(f"{name:<16}{got:>10,}{want:>10,}{s:>12}")
print("\npaper survival: dedup 13.3%, filter 92.3%")

# Free the LLM before Parler-TTS claims the GPU.
import torch
gc.collect(); torch.cuda.empty_cache()

## 5 · Push the text artifacts immediately

Checkpointing to the Hub *before* the long TTS run means a session timeout never costs the LLM stage.

In [ ]:
if STAGE in ("text", "both"):
    for man, cfg in [("bigrams_valid.jsonl", "bigrams"), ("sentences.jsonl", "sentences")]:
        run("csasr.data.push_to_hub", "--manifest", MAN / man,
            "--repo", SYNTH_REPO, "--config", cfg, "--text-only", "--token", HF_TOKEN)

## 6 · Stage 2b — synthesize audio

Indic Parler-TTS, Rohit (male) / Divya (female) 50/50. Emits 22.05 kHz → resampled to 16 kHz, written as int16 (float32 would be 7 GB).

Sharded: rerun this cell after a timeout and it skips whatever already exists.

In [ ]:
if STAGE in ("audio", "both"):
    # Pull sentences back from the Hub if this is a fresh session.
    if not (MAN / "sentences.jsonl").exists():
        ds = load_dataset(SYNTH_REPO, "sentences", split="train", token=HF_TOKEN)
        write_jsonl(MAN / "sentences.jsonl", [dict(r) for r in ds])

    for shard in range(NUM_SHARDS):
        run("csasr.tts.synthesize",
            "--sentences", MAN / "sentences.jsonl",
            "--audio-dir", AUDIO,
            "--out", MAN / f"train_t2.shard{shard}.jsonl",
            "--shard", str(shard), "--num-shards", str(NUM_SHARDS),
            "--batch-size", "8")

## 7 · Merge shards, build Train_T1 by reference

### GATE 2 — `Train_T1 ⊆ Train_T2`, durations ≈ 8h / 22h

In [ ]:
import itertools
merged = list(itertools.chain.from_iterable(
    read_jsonl(MAN / f"train_t2.shard{i}.jsonl") for i in range(NUM_SHARDS)
))
write_jsonl(MAN / "train_t2.jsonl", merged)
hours = sum(r["dur"] for r in merged) / 3600
print(f"Train_T2: {len(merged):,} clips, {hours:.2f} h (paper: 22 h)")

run("csasr.tts.make_subset", "--t2", MAN / "train_t2.jsonl",
    "--out", WORK / "t1_ids.json", "--hours", "8.0")

In [ ]:
# Spot-listen 2 clips per voice before committing 5 GPU-hours of training to them.
import IPython.display as ipd, random
from csasr.tts.speakers import assign_speaker
random.seed(0)
for voice in ("Rohit", "Divya"):
    picks = [r for r in merged if assign_speaker(r["sent_id"]).name == voice][:200]
    for r in random.sample(picks, 2):
        print(f"[{voice}] {r['text']}")
        ipd.display(ipd.Audio(r["wav"]))

## 8 · Push the synthetic corpus

Parquet + FLAC ≈ 1.4 GB (vs 2.5 GB as WAV). The round-trip check catches a corrupted upload *before* a 3h training run.

In [ ]:
run("csasr.data.push_to_hub", "--manifest", MAN / "train_t2.jsonl",
    "--repo", SYNTH_REPO, "--config", "synth_t2", "--token", HF_TOKEN, "--verify")

from huggingface_hub import HfApi
HfApi().upload_file(path_or_fileobj=str(WORK / "t1_ids.json"),
                    path_in_repo="t1_ids.json", repo_id=SYNTH_REPO,
                    repo_type="dataset", token=HF_TOKEN)
print("done ->", SYNTH_REPO)